# Step 3 — Blocking / Candidate Generation

Goal:
Generate a small set of candidate records for each Source-1 entity
without comparing every record against every other source.

Initial strategy:
1. Exact normalized business name
2. Country-aware matching
3. Measure candidate recall
4. Later add stronger blocking strategies

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

In [3]:
PROJECT_ROOT = Path.cwd().parent

DATASET_DIR = (
    PROJECT_ROOT
    / "student_resource"
    / "dataset"
)

TRAIN_DIR = DATASET_DIR / "train"

print("Project root:", PROJECT_ROOT)
print("Train directory:", TRAIN_DIR)

Project root: c:\Users\konda\23BQ1A4223\amazon-ml-challenge-2026
Train directory: c:\Users\konda\23BQ1A4223\amazon-ml-challenge-2026\student_resource\dataset\train


In [4]:
source1_path = TRAIN_DIR / "train_source1.tsv"

source1 = pd.read_csv(
    source1_path,
    sep="\t"
)

print("Source 1 shape:", source1.shape)

display(source1.head())

Source 1 shape: (2206821, 4)


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [5]:
print(list(TRAIN_DIR.glob("*.tsv")))

[WindowsPath('c:/Users/konda/23BQ1A4223/amazon-ml-challenge-2026/student_resource/dataset/train/train_ground_truth.tsv'), WindowsPath('c:/Users/konda/23BQ1A4223/amazon-ml-challenge-2026/student_resource/dataset/train/train_source1.tsv'), WindowsPath('c:/Users/konda/23BQ1A4223/amazon-ml-challenge-2026/student_resource/dataset/train/train_source2.tsv'), WindowsPath('c:/Users/konda/23BQ1A4223/amazon-ml-challenge-2026/student_resource/dataset/train/train_source3.tsv')]


In [6]:
source2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t"
)

source3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t"
)

print("Source 1:", source1.shape)
print("Source 2:", source2.shape)
print("Source 3:", source3.shape)

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)


In [7]:
sys.path.append(str(PROJECT_ROOT))

from src.normalization import add_normalized_columns

In [8]:
source1 = add_normalized_columns(source1)
source2 = add_normalized_columns(source2)
source3 = add_normalized_columns(source3)

In [9]:
print(source1.columns.tolist())

['entity_id', 'business_name', 'business_address', 'country', 'business_name_normalized', 'business_address_normalized', 'country_normalized']


In [10]:
source2["source_table"] = "source2"
source3["source_table"] = "source3"

candidates = pd.concat(
    [source2, source3],
    ignore_index=True
)

print("Total candidate records:", len(candidates))

display(candidates.head())

Total candidate records: 10320219


,entity_id,business_name,business_address,country,business_name_normalized,business_address_normalized,country_normalized,source_table
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,र म म र क ट ग प र इव ट ल म ट ड,kh no 570 13 new delhi west delhi delhi,india,source2
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,holloway peak incorporated seafood,105 elm street morganton nc,us,source2
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,आद त य प र पर ट ज एलएलप,g 3 571 gulmohar colony bhopal madhya pradesh,india,source2
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,summit incorporated,greensboro nc 19 1 2 stardust trail,us,source2
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,delta tetlecommunication incorporated,914 pierpont avenue cleveland oh,us,source2


In [11]:
candidates["block_key"] = (
    candidates["country_normalized"].fillna("")
    + "||"
    + candidates["business_name_normalized"].fillna("")
)

In [12]:
source1["block_key"] = (
    source1["country_normalized"].fillna("")
    + "||"
    + source1["business_name_normalized"].fillna("")
)

In [13]:
display(
    source1[
        [
            "entity_id",
            "business_name",
            "country",
            "block_key"
        ]
    ].head(10)
)

,entity_id,business_name,country,block_key
0,S1-925783039,Orelee's Barbershop,US,us||orelee s barbershop
1,S1-773889195,Prime Money,US,us||prime money
2,S1-377745466,B+ Retail Inc,US,us||b retail incorporated
3,S1-133037285,Christ Chapel,US,us||christ chapel
4,S1-755362802,Prabhav Business Center,India,india||prabhav business center
5,S1-851869949,Custom Wealth Services LLC,US,us||custom wealth services limited liability c...
6,S1-785847572,Consulting Nyasa Nursing Private Limited,India,india||consulting nyasa nursing private limited
7,S1-27541239,Nexus Anchor Rain,US,us||nexus anchor rain
8,S1-629417405,Moore Bitwise Inc,US,us||moore bitwise incorporated
9,S1-22305073,Dermatology Green Medicine,US,us||dermatology green medicine


In [14]:
ground_truth = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t"
)

print("Ground truth shape:", ground_truth.shape)

display(ground_truth.head())

Ground truth shape: (2206821, 2)


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [15]:
def parse_match_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    
    return [
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    ]


ground_truth["matched_entity_ids_list"] = (
    ground_truth["matched_entity_ids"]
    .apply(parse_match_ids)
)

ground_truth["match_count"] = (
    ground_truth["matched_entity_ids_list"].str.len()
)

print("Ground truth rows:", len(ground_truth))

print("\nMatch count statistics:")
print(ground_truth["match_count"].describe())

print(
    "\nZero-match Source-1 records:",
    (ground_truth["match_count"] == 0).sum()
)

print(
    "Source-1 records with at least one match:",
    (ground_truth["match_count"] > 0).sum()
)

Ground truth rows: 2206821

Match count statistics:
count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: match_count, dtype: float64

Zero-match Source-1 records: 123247
Source-1 records with at least one match: 2083574


In [16]:
# Create a compact lookup:
# country + normalized business name -> candidate entity IDs

candidate_lookup = (
    candidates[
        [
            "entity_id",
            "country_normalized",
            "business_name_normalized"
        ]
    ]
    .drop_duplicates()
)
print("Candidate lookup rows:", len(candidate_lookup))

Candidate lookup rows: 10320219


In [17]:
# Create the same blocking key for the candidate lookup
candidate_lookup["block_key"] = (
    candidate_lookup["country_normalized"].fillna("")
    + "||"
    + candidate_lookup["business_name_normalized"].fillna("")
)

# Create a lookup from block_key -> set of candidate entity IDs
block_to_ids = (
    candidate_lookup
    .groupby("block_key")["entity_id"]
    .agg(set)
)

print("Unique blocking keys:", len(block_to_ids))

Unique blocking keys: 7517697


In [18]:
block_sizes = (
    candidate_lookup["block_key"]
    .value_counts()
)

print("Total candidate records:", len(candidate_lookup))
print("Unique blocking keys:", len(block_sizes))

print("\nBlock size statistics:")
print(block_sizes.describe())

print(
    "\nBlocks containing exactly 1 record:",
    (block_sizes == 1).sum()
)

print(
    "Blocks containing more than 1 record:",
    (block_sizes > 1).sum()
)

print(
    "Largest block:",
    block_sizes.max()
)

Total candidate records: 10320219
Unique blocking keys: 7517697

Block size statistics:
count    7.517697e+06
mean     1.372790e+00
std      2.995698e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.042000e+03
Name: count, dtype: float64

Blocks containing exactly 1 record: 6505972
Blocks containing more than 1 record: 1011725
Largest block: 1042


In [19]:
# Create a fast lookup of block_key -> candidate entity IDs
candidate_pairs = candidate_lookup[
    ["block_key", "entity_id"]
].copy()

candidate_pairs = candidate_pairs.drop_duplicates()

# Convert each ground-truth row into its source-1 block key
source1_gt = source1[
    ["entity_id", "block_key"]
].merge(
    ground_truth[
        ["source1_entity_id", "matched_entity_ids_list"]
    ],
    left_on="entity_id",
    right_on="source1_entity_id",
    how="inner"
)

# Build a lookup for candidate IDs by block
block_to_candidate_ids = (
    candidate_pairs
    .groupby("block_key")["entity_id"]
    .apply(set)
)

# Check whether each true match is inside its block
def calculate_recall(row):
    candidate_ids = block_to_candidate_ids.get(
        row["block_key"],
        set()
    )

    true_ids = set(row["matched_entity_ids_list"])

    if not true_ids:
        return np.nan

    return len(true_ids & candidate_ids) / len(true_ids)


source1_gt["candidate_recall"] = source1_gt.apply(
    calculate_recall,
    axis=1
)

print(
    "Mean candidate recall:",
    source1_gt["candidate_recall"].mean()
)

print(
    "Perfect recall:",
    (source1_gt["candidate_recall"] == 1).mean()
)

print(
    "Zero recall:",
    (source1_gt["candidate_recall"] == 0).mean()
)

Mean candidate recall: 0.24628694333923187
Perfect recall: 0.031267601676801156
Zero recall: 0.38059588883738193


## Blocking Strategy 1 — Exact Normalized Name

Blocking key:

country + normalized business name

Results:

- Mean candidate recall: 24.63%
- Perfect recall: 3.13%
- Zero recall: 38.06%

Conclusion:

Exact normalized business name alone is insufficient.
Additional blocking strategies are required.

## Blocking Strategy 2 — Country + Name Token

Exact normalized name blocking misses records where business names
have small additions, removals, or word-order differences.

This block uses a strong token from the normalized business name
together with country to generate additional candidates.

In [20]:
def first_name_token(name):
    if pd.isna(name) or not str(name).strip():
        return ""
    
    tokens = str(name).split()
    
    for token in tokens:
        if len(token) >= 3:
            return token
    
    return tokens[0] if tokens else ""


candidates["name_token"] = candidates["business_name_normalized"].apply(
    first_name_token
)

source1["name_token"] = source1["business_name_normalized"].apply(
    first_name_token
)

candidates["token_block_key"] = (
    candidates["country_normalized"].fillna("")
    + "||"
    + candidates["name_token"]
)

source1["token_block_key"] = (
    source1["country_normalized"].fillna("")
    + "||"
    + source1["name_token"]
)

print("Candidate token blocks:", candidates["token_block_key"].nunique())
print("Source-1 token blocks:", source1["token_block_key"].nunique())

Candidate token blocks: 661006
Source-1 token blocks: 109563


In [21]:
token_block_sizes = candidates["token_block_key"].value_counts()

print("Token block statistics:")
print(token_block_sizes.describe())

print(
    "\nBlocks with exactly 1 candidate:",
    (token_block_sizes == 1).sum()
)

print(
    "Blocks with more than 1 candidate:",
    (token_block_sizes > 1).sum()
)

print(
    "Largest token block:",
    token_block_sizes.max()
)

Token block statistics:
count    661006.000000
mean         15.612898
std         392.421745
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max      126442.000000
Name: count, dtype: float64

Blocks with exactly 1 candidate: 464561
Blocks with more than 1 candidate: 196445
Largest token block: 126442


In [22]:
token_candidate_pairs = candidates[
    ["token_block_key", "entity_id"]
].drop_duplicates()

token_block_to_ids = (
    token_candidate_pairs
    .groupby("token_block_key")["entity_id"]
    .apply(set)
)

print(
    "Unique token blocks:",
    len(token_block_to_ids)
)

Unique token blocks: 661006


In [23]:
source1_gt_token = source1[
    ["entity_id", "token_block_key"]
].merge(
    ground_truth[
        ["source1_entity_id", "matched_entity_ids_list"]
    ],
    left_on="entity_id",
    right_on="source1_entity_id",
    how="inner"
)


def calculate_token_recall(row):
    candidate_ids = token_block_to_ids.get(
        row["token_block_key"],
        set()
    )

    true_ids = set(row["matched_entity_ids_list"])

    if not true_ids:
        return np.nan

    return len(true_ids & candidate_ids) / len(true_ids)


source1_gt_token["token_candidate_recall"] = (
    source1_gt_token.apply(
        calculate_token_recall,
        axis=1
    )
)


print(
    "Mean token-block recall:",
    source1_gt_token["token_candidate_recall"].mean()
)

print(
    "Perfect recall:",
    (
        source1_gt_token["token_candidate_recall"] == 1
    ).mean()
)

print(
    "Zero recall:",
    (
        source1_gt_token["token_candidate_recall"] == 0
    ).mean()
)

Mean token-block recall: 0.7329875394093176
Perfect recall: 0.38008112121463405
Zero recall: 0.04875746605637702


## Blocking Strategy 3 — Country + Address Token

Business names can differ between sources while the business address
remains similar.

This block uses the country and the first meaningful address token
to generate additional candidates.

In [24]:
# Extract the first meaningful address token (4+ characters)

candidates["address_token"] = (
    candidates["business_address_normalized"]
    .fillna("")
    .astype(str)
    .str.extract(r"\b([a-z0-9]{4,})\b", expand=False)
    .fillna("")
)

source1["address_token"] = (
    source1["business_address_normalized"]
    .fillna("")
    .astype(str)
    .str.extract(r"\b([a-z0-9]{4,})\b", expand=False)
    .fillna("")
)

# Create country + address token blocking key

candidates["address_block_key"] = (
    candidates["country_normalized"].fillna("")
    + "||"
    + candidates["address_token"]
)

source1["address_block_key"] = (
    source1["country_normalized"].fillna("")
    + "||"
    + source1["address_token"]
)

print(
    "Candidate address blocks:",
    candidates["address_block_key"].nunique()
)

print(
    "Source-1 address blocks:",
    source1["address_block_key"].nunique()
)

Candidate address blocks: 423213
Source-1 address blocks: 203990


In [25]:
address_block_sizes = (
    candidates["address_block_key"]
    .value_counts()
)

print("Address block statistics:")
print(address_block_sizes.describe())

print(
    "\nBlocks with exactly 1 candidate:",
    (address_block_sizes == 1).sum()
)

print(
    "Blocks with more than 1 candidate:",
    (address_block_sizes > 1).sum()
)

print(
    "Largest address block:",
    address_block_sizes.max()
)

Address block statistics:
count    423213.000000
mean         24.385402
std         732.470604
min           1.000000
25%           1.000000
50%           3.000000
75%           6.000000
max      222201.000000
Name: count, dtype: float64

Blocks with exactly 1 candidate: 144666
Blocks with more than 1 candidate: 278547
Largest address block: 222201


In [26]:
address_candidate_pairs = candidates[
    ["address_block_key", "entity_id"]
].drop_duplicates()

address_block_to_ids = (
    address_candidate_pairs
    .groupby("address_block_key")["entity_id"]
    .apply(set)
)

print(
    "Unique address blocks:",
    len(address_block_to_ids)
)

Unique address blocks: 423213


In [27]:
source1_gt_address = source1[
    ["entity_id", "address_block_key"]
].merge(
    ground_truth[
        ["source1_entity_id", "matched_entity_ids_list"]
    ],
    left_on="entity_id",
    right_on="source1_entity_id",
    how="inner"
)


def calculate_address_recall(row):
    candidate_ids = address_block_to_ids.get(
        row["address_block_key"],
        set()
    )

    true_ids = set(row["matched_entity_ids_list"])

    if not true_ids:
        return np.nan

    return len(true_ids & candidate_ids) / len(true_ids)


source1_gt_address["address_candidate_recall"] = (
    source1_gt_address.apply(
        calculate_address_recall,
        axis=1
    )
)


print(
    "Mean address-block recall:",
    source1_gt_address["address_candidate_recall"].mean()
)

print(
    "Perfect recall:",
    (
        source1_gt_address["address_candidate_recall"] == 1
    ).mean()
)

print(
    "Zero recall:",
    (
        source1_gt_address["address_candidate_recall"] == 0
    ).mean()
)

Mean address-block recall: 0.59530483062839
Perfect recall: 0.2704650717026891
Zero recall: 0.1544275679812726


## Combined Blocking Strategy

Final candidate generation combines:

1. Country + exact normalized business name
2. Country + first business-name token
3. Country + first address token

The union of these blocks should provide higher candidate recall
while keeping candidate generation manageable.

In [28]:
# Map each Source-1 record to candidates from each blocking strategy

source1["exact_candidates"] = source1["block_key"].map(
    block_to_candidate_ids
)

source1["token_candidates"] = source1["token_block_key"].map(
    token_block_to_ids
)

source1["address_candidates"] = source1["address_block_key"].map(
    address_block_to_ids
)

# Replace missing blocks with empty sets

source1["exact_candidates"] = source1["exact_candidates"].apply(
    lambda x: x if isinstance(x, set) else set()
)

source1["token_candidates"] = source1["token_candidates"].apply(
    lambda x: x if isinstance(x, set) else set()
)

source1["address_candidates"] = source1["address_candidates"].apply(
    lambda x: x if isinstance(x, set) else set()
)

print("Candidate sets created.")

Candidate sets created.


In [31]:
gt_pairs = ground_truth[
    ["source1_entity_id", "matched_entity_ids_list"]
].explode("matched_entity_ids_list")

gt_pairs = gt_pairs.rename(
    columns={"matched_entity_ids_list": "matched_entity_id"}
)

print("Ground-truth match pairs:", len(gt_pairs))

Ground-truth match pairs: 7761612


In [32]:
s1_keys = source1[
    [
        "entity_id",
        "block_key",
        "token_block_key",
        "address_block_key"
    ]
].rename(
    columns={"entity_id": "source1_entity_id"}
)

gt_pairs = gt_pairs.merge(
    s1_keys,
    on="source1_entity_id",
    how="left"
)

print("Pairs after Source-1 keys:", len(gt_pairs))

Pairs after Source-1 keys: 7761612


In [33]:
candidate_keys = candidates[
    [
        "entity_id",
        "block_key",
        "token_block_key",
        "address_block_key"
    ]
].drop_duplicates("entity_id")

candidate_keys = candidate_keys.rename(
    columns={"entity_id": "matched_entity_id"}
)

gt_pairs = gt_pairs.merge(
    candidate_keys,
    on="matched_entity_id",
    how="left",
    suffixes=("_s1", "_candidate")
)

print("Pairs after candidate keys:", len(gt_pairs))

Pairs after candidate keys: 7761612


In [34]:
gt_pairs["retrieved"] = (
    (gt_pairs["block_key_s1"] == gt_pairs["block_key_candidate"])
    |
    (gt_pairs["token_block_key_s1"] == gt_pairs["token_block_key_candidate"])
    |
    (gt_pairs["address_block_key_s1"] == gt_pairs["address_block_key_candidate"])
)

print(
    "Overall combined recall:",
    gt_pairs["retrieved"].mean()
)

entity_recall = (
    gt_pairs
    .groupby("source1_entity_id")["retrieved"]
    .mean()
)

print(
    "Mean candidate recall:",
    entity_recall.mean()
)

print(
    "Perfect recall:",
    (entity_recall == 1).mean()
)

print(
    "Zero recall:",
    (entity_recall == 0).mean()
)

Overall combined recall: 0.8795573651452817
Mean candidate recall: 0.8431420878449658
Perfect recall: 0.6728882859099129
Zero recall: 0.06896798607589831


In [35]:
missed_matches = gt_pairs[
    ~gt_pairs["retrieved"]
].copy()

print("Missed true matches:", len(missed_matches))

print(
    "Missed percentage:",
    len(missed_matches) / len(gt_pairs)
)

Missed true matches: 934829
Missed percentage: 0.12044263485471833


In [36]:
s1_info = source1[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "business_name_normalized",
        "business_address_normalized"
    ]
].rename(
    columns={"entity_id": "source1_entity_id"}
)

candidate_info = candidates[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "business_name_normalized",
        "business_address_normalized"
    ]
].rename(
    columns={"entity_id": "matched_entity_id"}
)

missed_sample = (
    missed_matches[
        [
            "source1_entity_id",
            "matched_entity_id"
        ]
    ]
    .head(1000)
    .merge(s1_info, on="source1_entity_id", how="left")
    .merge(candidate_info, on="matched_entity_id", how="left", suffixes=("_s1", "_candidate"))
)

display(
    missed_sample[
        [
            "source1_entity_id",
            "matched_entity_id",
            "business_name_s1",
            "business_name_candidate",
            "business_address_s1",
            "business_address_candidate",
            "country_s1",
            "country_candidate"
        ]
    ].head(30)
)

,source1_entity_id,matched_entity_id,business_name_s1,business_name_candidate,business_address_s1,business_address_candidate,country_s1,country_candidate
0,S1-965667,S3-775321672,Maure Williams Colombier Inc,Dréxkor,"85 Wayne Avenue, Ticonderoga, NY","85 Wanye Avenue, Ticonderoga Townshiip, New York",US,US
1,S1-656753428,S2-153058913,Ss Food Private Limited,एसएस फूड प्राइवेट लिमिटेड,"Af-684, Nandgram Near Mother India Public Scho...","AF-0684, NANDGRAM NEAR MOTHER INDIA PUBLIC SCH...",India,India
2,S1-656753428,S2-24659151,Ss Food Private Limited,एसएस फूड प्राइवेट लिमिटेड,"Af-684, Nandgram Near Mother India Public Scho...","AF-0684, Uttar Pradesh, GHAZIABAD, 9487203",India,India
3,S1-656753428,S3-679606215,Ss Food Private Limited,एसएस फूड प्राइवेट लिमिटेड,"Af-684, Nandgram Near Mother India Public Scho...","Af-684, Ghaziabad, UP",India,India
4,S1-318373630,S2-660036492,Red Ventures Private Limited,रेड वेंचर्स प्राइवेट लिमिटेड,"Rajasthan, Jaipur, Banipark, Gokul Apartment, ...","G-1, BANIPARK, JAIPUR, Rajasthan",India,India
5,S1-7293388,S3-523120965,Chordia & Partners,Smt Chordia & Center,"Faridabad, 1038 Sector 9, Haryana","#1038 Sector 9, Faridabad, हरियाणा",India,India
6,S1-546142636,S3-200008747,Crystal Staffing Solutions LLC,Llc Crystal Staffing Solutions,"8706 Kentucky Derby Drive, Waxhaw, NC","870 Kentucky Derby Drive, Waxhaw, North Carolina",US,US
7,S1-546142636,S3-729771680,Crystal Staffing Solutions LLC,LLC Crystal Shaffing Solutions,"8706 Kentucky Derby Drive, Waxhaw, NC","870 Kentucky Derby Drive, Waxhaw, North Carolina",US,US
8,S1-145361722,S3-96572514,Dick Regional Armada Corp,[Corp] Dick Regional Armada,"33466 Warwick Hills Road, Yucaipa, CA","Yucaipa, California, 33466 Warwik Hills Road",US,US
9,S1-561341312,S2-483615364,Balaji Investment Private Limited,బాలాజీ ఇన్వెస్ట్‌మెంట్ ప్రైవేట్ లిమిటెడ్,"Plot No. D-88 & D-90, Hyd, Telangana, Hyderaba...","H.NO 00516 PLOT NO. D-88 & D-90, JEEDIMETLA, H...",India,India


In [38]:
def token_overlap(name1, name2):
    a = set(str(name1).split())
    b = set(str(name2).split())

    if not a or not b:
        return 0.0

    return len(a & b) / len(a | b)


missed_sample["name_token_overlap"] = missed_sample.apply(
    lambda row: token_overlap(
        row["business_name_normalized_s1"],
        row["business_name_normalized_candidate"]
    ),
    axis=1
)

print(
    missed_sample["name_token_overlap"].describe()
)

count    1000.000000
mean        0.253325
std         0.356012
min         0.000000
25%         0.000000
50%         0.000000
75%         0.600000
max         1.000000
Name: name_token_overlap, dtype: float64


In [39]:
# Block 4: Country + first 4 characters of business name

candidates["name_prefix4"] = (
    candidates["business_name_normalized"]
    .fillna("")
    .astype(str)
    .str.replace(" ", "", regex=False)
    .str[:4]
)

source1["name_prefix4"] = (
    source1["business_name_normalized"]
    .fillna("")
    .astype(str)
    .str.replace(" ", "", regex=False)
    .str[:4]
)

candidates["prefix_block_key"] = (
    candidates["country_normalized"].fillna("")
    + "||"
    + candidates["name_prefix4"]
)

source1["prefix_block_key"] = (
    source1["country_normalized"].fillna("")
    + "||"
    + source1["name_prefix4"]
)

print(
    "Candidate prefix blocks:",
    candidates["prefix_block_key"].nunique()
)

print(
    "Source-1 prefix blocks:",
    source1["prefix_block_key"].nunique()
)

Candidate prefix blocks: 298149
Source-1 prefix blocks: 129451


In [40]:
prefix_block_sizes = candidates["prefix_block_key"].value_counts()

print("Prefix block statistics:")
print(prefix_block_sizes.describe())

print(
    "\nBlocks with exactly 1 candidate:",
    (prefix_block_sizes == 1).sum()
)

print(
    "Blocks with more than 1 candidate:",
    (prefix_block_sizes > 1).sum()
)

print(
    "Largest prefix block:",
    prefix_block_sizes.max()
)

Prefix block statistics:
count    298149.000000
mean         34.614300
std         556.436609
min           1.000000
25%           1.000000
50%           2.000000
75%           6.000000
max      116596.000000
Name: count, dtype: float64

Blocks with exactly 1 candidate: 115202
Blocks with more than 1 candidate: 182947
Largest prefix block: 116596


In [41]:
prefix_candidate_pairs = candidates[
    ["prefix_block_key", "entity_id"]
].drop_duplicates()

prefix_block_to_ids = (
    prefix_candidate_pairs
    .groupby("prefix_block_key")["entity_id"]
    .apply(set)
)

print(
    "Unique prefix blocks:",
    len(prefix_block_to_ids)
)

Unique prefix blocks: 298149


In [42]:
source1_gt_prefix = source1[
    ["entity_id", "prefix_block_key"]
].merge(
    ground_truth[
        ["source1_entity_id", "matched_entity_ids_list"]
    ],
    left_on="entity_id",
    right_on="source1_entity_id",
    how="inner"
)


def calculate_prefix_recall(row):
    candidate_ids = prefix_block_to_ids.get(
        row["prefix_block_key"],
        set()
    )

    true_ids = set(row["matched_entity_ids_list"])

    if not true_ids:
        return np.nan

    return len(true_ids & candidate_ids) / len(true_ids)


source1_gt_prefix["prefix_candidate_recall"] = (
    source1_gt_prefix.apply(
        calculate_prefix_recall,
        axis=1
    )
)

print(
    "Mean prefix-block recall:",
    source1_gt_prefix["prefix_candidate_recall"].mean()
)

print(
    "Perfect recall:",
    (
        source1_gt_prefix["prefix_candidate_recall"] == 1
    ).mean()
)

print(
    "Zero recall:",
    (
        source1_gt_prefix["prefix_candidate_recall"] == 0
    ).mean()
)

Mean prefix-block recall: 0.7605292570108795
Perfect recall: 0.43965958272102723
Zero recall: 0.04642923010067423
